# BigQuery: Conversational Analytics with Gemini (March 2026 Suite)

[![Open In Colab](https://colab.research.google.com/github/maruti123/partner-demos/blob/main/partner-demos-march-2026/bq_conversational_analytics_demo.ipynb)](https://colab.research.google.com/github/maruti123/partner-demos/blob/main/partner-demos-march-2026/bq_conversational_analytics_demo.ipynb)

**What you'll see:** A high-fidelity data agent that leverages the **BigQuery Conversational Analytics API** (`ask_data_insights`) to provide an intelligent, multi-turn data exploration experience, including native **ObjectRef** support for unstructured data.

## Scenario: The Intelligent Data Partner
A retail analyst uses the agent to perform an investigation. The agent uses the specialized Data AI backend to handle complex BQML logic and unstructured data audit seamlessly.

| Phase | Feature | Agent Behavior |
|------|---------|----------------|
| **Discovery** | `ask_data_insights` | **Core Highlight**: Agent calls the CA API to find and explain anomalies in sales |
| **Deep-Dive** | **ObjectRef** | Agent leverages CA API to join structured records with GCS object metadata |
| **Planning** | `AI.FORECAST` & `AI.DETECT_ANOMALIES` | Agent uses native reasoning to predict trends and flag future outliers |

### Key Technologies
- **BigQuery Conversational Analytics API** — Specialized backend for high-accuracy NL-to-insights transformations
- **ObjectRef** — Native integration between Conversational Analytics and Cloud Storage for auditing PDFs/images
- **BigQueryToolset** — Standard ADK toolset containing the `ask_data_insights` tool
- **Gemini 3.1 Pro** (Preview) — Powering the high-level reasoning and tool orchestration

### Requirements
- `google-adk >= 1.28.0` and `google-genai >= 1.69.0`
- BigQuery, Gemini Data Analytics, and Vertex AI APIs enabled
- A Google Cloud project with billing enabled

In [ ]:
# 1. Setup and Authentication
%pip install "google-adk>=1.28.0" google-genai google-cloud-bigquery google-cloud-storage nest-asyncio --quiet --index-url https://pypi.org/simple

try:
    from google.colab import auth
    auth.authenticate_user()
    print('Authenticated via Colab')
except ModuleNotFoundError:
    print('Not running in Colab — using Application Default Credentials (ADC)')

import os
import nest_asyncio
import time
import google.auth
nest_asyncio.apply()

project_id = 'YOUR_PROJECT_ID'  # @param {type:"string"}
user_email = 'YOUR_EMAIL@google.com' # @param {type:"string"}
location = 'US' # @param {type:"string"}

os.environ["GOOGLE_CLOUD_PROJECT"] = project_id
os.environ["GOOGLE_CLOUD_LOCATION"] = location
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"

In [ ]:
# 2. Enable APIs
!gcloud services enable bigquery.googleapis.com geminidataanalytics.googleapis.com storage.googleapis.com aiplatform.googleapis.com cloudaicompanion.googleapis.com --project={project_id} --quiet
print("APIs enabled.")

In [ ]:
# 3. Configure IAM Roles (Zero-Console)
roles = [
    "roles/bigquery.connectionAdmin",
    "roles/bigquery.admin",
    "roles/resourcemanager.projectIamAdmin",
    "roles/geminidataanalytics.dataAgentUser",
    "roles/cloudaicompanion.user"
]

for role in roles:
    print(f"Granting {role} to {user_email}...")
    !gcloud projects add-iam-policy-binding {project_id} --member=user:{user_email} --role={role} --condition=None --quiet > /dev/null

print("\nSuccess: IAM Roles configured. Waiting for propagation...")
time.sleep(30)

In [ ]:
from google.cloud import bigquery, storage
from datetime import datetime, timedelta
import random
import json
import re

def setup_infrastructure():
    bq_client = bigquery.Client(project=project_id, location=location)
    storage_client = storage.Client(project=project_id)

    dataset_id = f"{project_id}.march_demo"
    dataset = bigquery.Dataset(dataset_id)
    dataset.location = location
    bq_client.create_dataset(dataset, exists_ok=True)
    print(f"[1/5] Dataset '{dataset_id}' ready.")

    # 1. Create Sales Table
    table_id = f"{dataset_id}.sales_data"
    schema = [
        bigquery.SchemaField("sale_date", "DATE"),
        bigquery.SchemaField("product_id", "STRING"),
        bigquery.SchemaField("amount", "FLOAT"),
        bigquery.SchemaField("receipt_id", "STRING"),
    ]
    random.seed(42)
    data = [
        {"sale_date": (datetime.now() - timedelta(days=i)).date().isoformat(),
         "product_id": f"PROD_{i%5}",
         "amount": round(200.0 + random.uniform(-50, 50), 2),
         "receipt_id": f"R_{1000+i}"}
        for i in range(365)
    ]
    data.append({"sale_date": datetime.now().date().isoformat(),
                 "product_id": "PROD_999", "amount": 20000.0, "receipt_id": "R_ANOMALY"})
    
    job_config = bigquery.LoadJobConfig(
        schema=schema, write_disposition="WRITE_TRUNCATE",
        time_partitioning=bigquery.TimePartitioning(field="sale_date"),
    )
    load_job = bq_client.load_table_from_json(data, table_id, job_config=job_config)
    load_job.result()
    print(f"[2/5] Table '{table_id}' loaded.")

    # 2. Setup GCS Bucket & Receipt
    bucket_name = f"{project_id}-receipts"
    bucket = (storage_client.create_bucket(bucket_name, location=location)
              if not storage_client.lookup_bucket(bucket_name)
              else storage_client.get_bucket(bucket_name))
    blob = bucket.blob("receipts/R_ANOMALY.jpg")
    blob.upload_from_string(b"Dummy receipt binary", content_type="image/jpeg")
    print(f"[3/5] GCS bucket 'gs://{bucket_name}' ready.")

    # 3. Create BigQuery Connection for Object Tables
    conn_id = "gcs-conn"
    # Fixed: Flag positioning for --quiet
    !bq mk --quiet --connection --location={location} --project_id={project_id} --connection_type=CLOUD_RESOURCE {conn_id}
    
    # Authorize the connection's service account (Robustly)
    print("[4/5] Authorizing connection service account...")
    out = get_ipython().getoutput(f'bq --quiet --project_id={project_id} show --connection --location={location} --format=json {conn_id}')
    try:
        json_str = re.search(r'\{.*\}', "".join(out), re.DOTALL).group(0)
        conn_info = json.loads(json_str)
        conn_sa = conn_info["cloudResource"]["serviceAccountId"]
        !gcloud projects add-iam-policy-binding {project_id} --member=serviceAccount:{conn_sa} --role=roles/storage.objectViewer --condition=None --quiet > /dev/null
        print(f"      Success: Connection '{conn_id}' authorized.")
    except Exception as e:
        print(f"      Error authorizing connection: {e}")
    
    print(f"Waiting 15s for propagation...")
    time.sleep(15)

    # 4. Create EXTERNAL OBJECT TABLE
    object_table_id = f"{dataset_id}.receipt_objects"
    sql = f"""
    CREATE OR REPLACE EXTERNAL TABLE `{object_table_id}`
    WITH CONNECTION `{project_id}.{location}.{conn_id}`
    OPTIONS (
      object_metadata = 'SIMPLE',
      uris = ['gs://{bucket_name}/receipts/*']
    )"""
    bq_client.query(sql).result()
    print(f"[5/5] External Object Table '{object_table_id}' created for ObjectRef queries.")

setup_infrastructure()

In [ ]:
from google.adk import Agent, Runner
from google.adk.sessions.in_memory_session_service import InMemorySessionService
from google.adk.tools.bigquery import BigQueryToolset, BigQueryCredentialsConfig
from google.adk.tools.bigquery.config import BigQueryToolConfig
from google.genai import types
import google.auth

# 1. Get existing credentials from the environment
creds, _ = google.auth.default()

# 2. Initialize the BQ Toolset (Filtered for pure Conversational Analytics)
# We exclude 'forecast' and 'detect_anomalies' to force the use of 'ask_data_insights'.
bq_toolset = BigQueryToolset(
    tool_filter=["ask_data_insights", "get_table_info", "get_dataset_info", "list_table_ids", "execute_sql"],
    credentials_config=BigQueryCredentialsConfig(credentials=creds),
    bigquery_tool_config=BigQueryToolConfig(
        job_labels={"ca-bq-job": "true", "demo": "march-2026-suite"}
    )
)

os.environ["GOOGLE_CLOUD_LOCATION"] = "global"  # Gemini 3.1 Pro Preview endpoint

fq_table = f"{project_id}.march_demo.sales_data"
fq_obj_table = f"{project_id}.march_demo.receipt_objects"

# 3. Define the Conversational Analytics Agent
agent = Agent(
    model="gemini-3.1-pro-preview",
    name="DataInsightsAssistant",
    instruction=f"""You are an interactive data analyst specializing in BigQuery.
    
    CRITICAL: For ALL natural language data questions, forecasts, anomaly checks, 
    and ObjectRef-based audits (joining structured sales with GCS receipts), 
    you MUST use the `ask_data_insights` tool. This tool calls the specialized 
    Conversational Analytics API and is your ONLY interface for reasoning about data.
    
    Do not attempt to write complex SQL manually for ObjectRef audits; use `ask_data_insights` 
    to handle the cross-modal reasoning between `{fq_table}` and `{fq_obj_table}` natively.
    
    Present all results in clear table format. Stay in the conversation to handle follow-up questions.""",
    tools=[bq_toolset]
)

runner = Runner(
    agent=agent,
    session_service=InMemorySessionService(),
    app_name="conversational_analytics_demo",
    auto_create_session=True
)

async def run_agent(prompt: str):
    print(f"User: {prompt}\n")
    message = types.Content(parts=[types.Part(text=prompt)], role='user')
    async for event in runner.run_async(
        user_id="partner_user", session_id="march_session", new_message=message
    ):
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    print(f"Agent: {part.text}")
                if part.function_call:
                    print(f"  >> [SYSTEM]: Calling Tool '{part.function_call.name}'")

print("Data AI Agent ready.")

In [ ]:
await run_agent("Check my sales data for any anomalies in the daily totals this month.")

In [ ]:
await run_agent("For the anomalous sale with receipt_id 'R_ANOMALY', find the corresponding receipt URI from the object table so I can audit it.")

In [ ]:
await run_agent("Now, assuming that anomaly was resolved, forecast the daily sales for the next 7 days using the CA API.")

### 6. Key Takeaways

- **`ask_data_insights`** is the recommended primary interface for the 2026 unified conversational experience.
- **`ObjectRef`** support in Conversational Analytics bridges the gap between structured records and unstructured GCS files without moving data.